In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# relocate the notebook
import sys
import os

sys.path.append(os.path.abspath("..")) # only for imports
os.chdir("..") # move working directory up one level
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"  # allocate on demand

In [3]:
!pwd

/home/julien-rsbrg/Documents/thesis/Games/FreeChoiceLearningTasksPrototypes/Railway-GeneralizationGridTask


# Imports

In [4]:
import pandas as pd
import numpy as np

# Questionnaires

In [6]:
q = pd.read_csv("data/preprocessed/per_session/record_20261007_171538_questionnaire.csv")
q_cols = ["session_id","language","administration","questionnaire_id","question","response"]
q[q_cols]

,session_id,language,administration,questionnaire_id,question,response
0,01c9295b_2026-09-29_15-06-22,FR,1,ChildOCQ,donbura,assez familier
1,01c9295b_2026-09-29_15-06-22,FR,1,ChildOCQ,vlépa,modérément familier
2,01c9295b_2026-09-29_15-06-22,FR,1,ChildOCQ,brocomme,modérément familier
3,01c9295b_2026-09-29_15-06-22,FR,1,ChildOCQ,daredevil,modérément familier
4,01c9295b_2026-09-29_15-06-22,FR,1,ChildOCQ,elsa,modérément familier
...,...,...,...,...,...,...
863,dfb95fe0_2026-09-29_09-14-28,FR,1,JrMAI,Je vérifie de temps en temps que je vais bien ...,Souvent
864,dfb95fe0_2026-09-29_09-14-28,FR,1,JrMAI,Je détermine ce que je dois accomplir avant de...,Parfois
865,dfb95fe0_2026-09-29_09-14-28,FR,1,JrMAI,Je réfléchis à ce que je dois apprendre avant ...,Rarement
866,dfb95fe0_2026-09-29_09-14-28,FR,1,JrMAI,Je sais quand je comprends quelque chose.,Parfois


# Exploration

In [7]:
N_COLUMNS, N_ROWS = 7, 7

In [8]:
e = pd.read_csv("data/preprocessed/per_session/record_20261007_171538_exploration.csv")
e["action"] = e["position_rowID"]*N_COLUMNS + e["position_columnID"]

In [9]:
e.columns.tolist()

['session_id',
 'prolific_pid',
 'prolific_session',
 'trial',
 'position_rowID',
 'position_columnID',
 'position_x',
 'position_y',
 'predicted_value',
 'actual_value',
 'language',
 'context',
 'client_time',
 'received_at',
 'study_id',
 'only_select',
 'action']

In [10]:
restricted_e = e[~e["only_select"]][["session_id","trial","action","predicted_value","actual_value"]]
restricted_e["trial_type"] = "free_choice"
restricted_e.loc[restricted_e["trial"] < 2, "trial_type"] = "forced_choice"
restricted_e.rename(columns={"predicted_value":"p_action"}, inplace=True)

# retrieve s_{i} with i env.n_actions
N_ENV_ACTIONS = 49
for i in range(N_ENV_ACTIONS):
    name = f"s_{i}"
    restricted_e[name] = np.nan

    mask_obs = restricted_e["action"] == i
    restricted_e.loc[mask_obs,name] = restricted_e.loc[mask_obs,"actual_value"]


restricted_e

,session_id,trial,action,p_action,actual_value,trial_type,s_0,s_1,s_2,s_3,...,s_39,s_40,s_41,s_42,s_43,s_44,s_45,s_46,s_47,s_48
4,1251a7db_2026-10-01_13-09-04,0,26,5,4,forced_choice,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1251a7db_2026-10-01_13-09-04,1,26,3,2,forced_choice,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1251a7db_2026-10-01_13-09-04,2,48,6,2,free_choice,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
10,1251a7db_2026-10-01_13-09-04,3,47,3,1,free_choice,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
12,1251a7db_2026-10-01_13-09-04,4,41,2,4,free_choice,NaN,NaN,NaN,NaN,...,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271,dfb95fe0_2026-09-29_09-14-28,5,25,9,5,free_choice,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
273,dfb95fe0_2026-09-29_09-14-28,6,19,9,7,free_choice,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
275,dfb95fe0_2026-09-29_09-14-28,7,8,5,10,free_choice,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
277,dfb95fe0_2026-09-29_09-14-28,8,29,9,5,free_choice,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# full trails are 62
restricted_e.groupby("session_id")["trial"].count()

session_id
1251a7db_2026-10-01_13-09-04    62
8b42342d_2026-10-01_14-37-42    62
dfb95fe0_2026-09-29_09-14-28    10
Name: trial, dtype: int64

In [12]:
dst_folder = "data/preprocessed/per_participant"
for s_id in ["1251a7db_2026-10-01_13-09-04","8b42342d_2026-10-01_14-37-42"]:
    mask = restricted_e["session_id"] == s_id
    restricted_e[mask].reset_index(drop=True).to_csv(os.path.join(dst_folder,s_id+".csv"))